# Слой FEATURES: признаки для модели

**Вход:** ../processes/building_violations_processed.parquet — одна строка = одна инспекция.

Задача модели: заранее оценить, провалит ли объект инспекцию (target_failed = 1, если inspection_status = FAILED). Поэтому берём только то, что известно до инспекции: календарь, служба и тип проверки, место, история объекта и инспектора. История считается строго по прошлым инспекциям, чтобы не было утечки будущего.

In [1]:
import json
from pathlib import Path

import pandas as pd

processed = pd.read_parquet('../processes/building_violations_processed.parquet')
print(processed.shape)
processed.head()

(470057, 18)


,inspection_number,inspection_date,last_modified,property_group,address,department_bureau,inspection_category,inspector_id,inspection_waived,latitude,longitude,n_violations,n_unique_codes,n_open,n_complied,days_to_status_mean,inspection_status,share_open
0,9647125,2007-11-04,2002-10-31,3021,800 N STATE ST,ELEVATOR,PERMIT,519853,N,41.896717,-87.628349,4,1,4,0,NaN,PASSED,1.0
1,520157,2007-09-18,2003-10-14,2688,55 E ERIE ST,ELEVATOR,PERMIT,519853,N,41.893997,-87.626030,1,1,1,0,NaN,PASSED,1.0
2,535353,2007-12-15,2003-12-23,1821,230 N MICHIGAN AVE,ELEVATOR,PERMIT,541663,N,41.886517,-87.624680,1,1,1,0,NaN,FAILED,1.0
3,1124236,2007-05-01,2005-08-08,547063,1655 S CALIFORNIA BLVD,CONSERVATION,COMPLAINT,BL00340,N,41.857597,-87.695496,1,1,1,0,NaN,FAILED,1.0
4,1128275,2006-01-05,2005-08-15,22634,7250 S LOWE AVE,CONSERVATION,COMPLAINT,BL00876,N,41.761948,-87.640152,12,12,12,0,NaN,FAILED,1.0


## 1. Целевая метка и календарь

In [2]:
df = processed[processed['inspection_date'].notna()]
df = df.sort_values(['inspection_date', 'inspection_number'], kind='stable').reset_index(drop=True)

df['target_failed'] = df['inspection_status'].astype('string').eq('FAILED').fillna(False).astype('int8')

d = df['inspection_date']
df['year'] = d.dt.year.astype('int16')
df['month'] = d.dt.month.astype('int8')
df['dayofweek'] = d.dt.dayofweek.astype('int8')
df['heating_season'] = d.dt.month.isin([10, 11, 12, 1, 2, 3, 4, 5]).astype('int8')

print(df['target_failed'].value_counts(normalize=True).round(3))

target_failed
1    0.628
0    0.372
Name: proportion, dtype: float64


## 2. История объекта

Объект определяется по property group: так город связывает разные адреса одного участка. Считаем, сколько раз объект проверяли раньше, какая доля прошлых проверок провалена и сколько дней прошло с прошлой инспекции.

In [3]:
obj_key = df['property_group'].astype('string').fillna('NA_' + df['inspection_number'].astype('string'))
g = df.groupby(obj_key, sort=False)

df['obj_prev_inspections'] = g.cumcount().astype('int32')
prev_fails = g['target_failed'].cumsum() - df['target_failed']
has_history = df['obj_prev_inspections'].where(df['obj_prev_inspections'] > 0)
df['obj_prev_fail_rate'] = (prev_fails / has_history).astype('float32')
df['obj_days_since_prev'] = g['inspection_date'].diff().dt.days.astype('float32')
df['obj_is_new'] = (df['obj_prev_inspections'] == 0).astype('int8')

df[['obj_prev_inspections', 'obj_prev_fail_rate', 'obj_days_since_prev']].describe().round(2)

,obj_prev_inspections,obj_prev_fail_rate,obj_days_since_prev
count,470057.00,314353.00,314353.00
mean,3.65,0.66,716.88
std,7.45,0.36,1068.66
min,0.00,0.00,0.00
25%,0.00,0.48,13.00
50%,1.00,0.75,272.00
75%,4.00,1.00,933.00
max,272.00,1.00,7485.00


## 3. История инспектора

In [4]:
insp_key = df['inspector_id'].astype('string').fillna('UNKNOWN')
gi = df.groupby(insp_key, sort=False)

df['insp_prev_inspections'] = gi.cumcount().astype('int32')
insp_prev_fails = gi['target_failed'].cumsum() - df['target_failed']
insp_history = df['insp_prev_inspections'].where(df['insp_prev_inspections'] > 0)
df['insp_prev_fail_rate'] = (insp_prev_fails / insp_history).astype('float32')

## 4. Отбор признаков

Число найденных нарушений и доля открытых известны только после инспекции. В признаки для прогноза их не берём, иначе модель подсмотрит ответ.

In [5]:
FEATURES = [
    'department_bureau', 'inspection_category', 'inspection_waived',
    'year', 'month', 'dayofweek', 'heating_season',
    'latitude', 'longitude',
    'obj_prev_inspections', 'obj_prev_fail_rate', 'obj_days_since_prev', 'obj_is_new',
    'insp_prev_inspections', 'insp_prev_fail_rate',
]
KEYS = ['inspection_number', 'inspection_date']
TARGET = ['target_failed', 'inspection_status']

features = df[KEYS + FEATURES + TARGET]
features.to_parquet('building_violations_features.parquet', index=False)

print('Features созданы!')
print('строк:', f'{len(features):,}', '| признаков:', len(FEATURES))

Features созданы!
строк: 470,057 | признаков: 15


## 5. Проверка: признаки связаны с целевой меткой

In [6]:
print('Доля FAILED по службам:')
print(df.groupby('department_bureau', observed=True)['target_failed'].mean()
        .sort_values(ascending=False).round(3).head(10))
print()
print('Доля FAILED в зависимости от истории объекта:')
bins = pd.cut(df['obj_prev_fail_rate'], [-0.01, 0.25, 0.5, 0.75, 1.0])
print(df.groupby(bins, observed=True)['target_failed'].mean().round(3))
print('новые объекты без истории:', round(float(df.loc[df['obj_is_new'] == 1, 'target_failed'].mean()), 3))

Доля FAILED по службам:
department_bureau
SIGNS                         0.910
CONSERVATION                  0.820
NEW CONSTRUCTION              0.691
IRON                          0.623
SPECIAL TASK FORCE            0.603
REFRIGERATION                 0.597
SPECIAL INSPECTION PROGRAM    0.571
ELEVATOR                      0.549
PLUMBING                      0.514
ELECTRICAL                    0.498
Name: target_failed, dtype: float64

Доля FAILED в зависимости от истории объекта:
obj_prev_fail_rate
(-0.01, 0.25]    0.387
(0.25, 0.5]      0.549
(0.5, 0.75]      0.645
(0.75, 1.0]      0.702
Name: target_failed, dtype: float64
новые объекты без истории: 0.667


In [7]:
feature_stats = {
    'rows': int(len(features)),
    'n_features': len(FEATURES),
    'features': FEATURES,
    'target_failed_share': round(float(features['target_failed'].mean()), 4),
    'features_mb': round(Path('building_violations_features.parquet').stat().st_size / 1e6, 2),
}
Path('feature_stats.json').write_text(json.dumps(feature_stats, ensure_ascii=False, indent=2), encoding='utf-8')
features.head()

,inspection_number,inspection_date,department_bureau,inspection_category,inspection_waived,year,month,dayofweek,heating_season,latitude,longitude,obj_prev_inspections,obj_prev_fail_rate,obj_days_since_prev,obj_is_new,insp_prev_inspections,insp_prev_fail_rate,target_failed,inspection_status
0,1096365,2006-01-01,REFRIGERATION,PERIODIC,N,2006,1,6,1,41.999992,-87.815224,0,NaN,NaN,1,0,NaN,1,FAILED
1,1104785,2006-01-01,CONSERVATION,PERIODIC,N,2006,1,6,1,41.786442,-87.684441,0,NaN,NaN,1,0,NaN,1,FAILED
2,1125530,2006-01-01,CONSERVATION,PERIODIC,N,2006,1,6,1,41.877460,-87.749161,0,NaN,NaN,1,0,NaN,1,FAILED
3,1128540,2006-01-01,CONSERVATION,PERIODIC,N,2006,1,6,1,41.920471,-87.710487,0,NaN,NaN,1,0,NaN,1,FAILED
4,1131620,2006-01-01,CONSERVATION,PERIODIC,N,2006,1,6,1,41.778713,-87.615738,0,NaN,NaN,1,0,NaN,1,FAILED
